In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm import tqdm

print("Carregando os dados temporais...")
# Certifique-se de que os dados estão ordenados temporalmente antes de tudo!
# df = df.sort_values(['TAG', 'Data_Evento']) 
df = pd.read_csv('dados_lstm.csv')

# 1. Tratamento de Categóricas (O LabelEncoder pode ser aplicado em todo o dataset 
# para mapear o vocabulário, desde que não vaze informações estatísticas do target)
le_alarme = LabelEncoder()
le_operador = LabelEncoder()
df['Id_Alarme'] = le_alarme.fit_transform(df['Id_Alarme'])
df['Matricula_Operador_Hash'] = le_operador.fit_transform(df['Matricula_Operador_Hash'])

# Salvando o tamanho do vocabulário para as camadas de Embedding
VOCAB_ALARME = df['Id_Alarme'].max() + 1
VOCAB_OPERADOR = df['Matricula_Operador_Hash'].max() + 1

# 2. Separação Cronológica de Treino (80%) e Teste (20%) agrupada por TAG
print("Realizando Split Cronológico por TAG...")
train_list, test_list = [], []

for tag, group in tqdm(df.groupby('TAG'), desc="Splitting"):
    # Split de 80% preservando a ordem do tempo
    split_idx = int(len(group) * 0.8)
    train_list.append(group.iloc[:split_idx])
    test_list.append(group.iloc[split_idx:])

df_train = pd.concat(train_list)
df_test = pd.concat(test_list)

# 3. Padronização SEM DATA LEAKAGE
# O fit_transform ocorre APENAS no treino. O test recebe apenas transform.
print("Aplicando StandardScaler (Fit no Treino, Transform no Teste)...")
scaler = StandardScaler()
cols_to_scale = ['Valor', 'Tempo_Trabalhando_Horas']

df_train.loc[:, cols_to_scale] = scaler.fit_transform(df_train[cols_to_scale])
df_test.loc[:, cols_to_scale] = scaler.transform(df_test[cols_to_scale])

# 4. Configurações da Janela Temporal
cols_features = [
    'Tag_Frota', 'Tipo', 'Matricula_Operador_Hash', 'Id_Alarme', 'Id_Criticidade', 
    'Valor', 'Tempo_Trabalhando_Horas', 'Periodo_Dia_Int', 
    'Dia_1', 'Dia_2', 'Dia_3', 'Dia_4', 'Dia_5', 'Dia_6'
]
SEQ_LENGTH = 30
PROBABILIDADE_MANTER_ZERO = 0.30  # Ajustado para manter 30% no treino

def criar_tensores(df_subset, is_train=True):
    X_list, y_list = [], []
    
    for tag, group in tqdm(df_subset.groupby('TAG'), desc=f"Gerando janelas ({'Treino' if is_train else 'Teste'})"):
        X_group = group[cols_features].values
        y_group = group['Target_4h'].values
        
        if len(X_group) < SEQ_LENGTH:
            continue
            
        for i in range(len(X_group) - SEQ_LENGTH):
            target = y_group[i + SEQ_LENGTH - 1]
            
            # Sub-amostragem da classe majoritária APENAS no conjunto de treinamento
            if is_train and target == 0 and np.random.rand() > PROBABILIDADE_MANTER_ZERO:
                continue
                
            janela = X_group[i : i + SEQ_LENGTH]
            X_list.append(janela)
            y_list.append(target)
            
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int8)

# Gerando os Tensores Finais
print("\nGerando tensores de Treino...")
X_train, y_train = criar_tensores(df_train, is_train=True)

print("\nGerando tensores de Teste...")
X_test, y_test = criar_tensores(df_test, is_train=False)

print("\n--- RESUMO DOS TENSORES ---")
print(f"Treino X: {X_train.shape} | Y: {y_train.shape} | % Classe 1: {(y_train.mean()*100):.2f}%")
print(f"Teste  X: {X_test.shape}  | Y: {y_test.shape}  | % Classe 1: {(y_test.mean()*100):.2f}%")

# Salvando
np.save('X_train.npy', X_train)
np.save('y_train.npy', y_train)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)
print("Tensores isolados e salvos com sucesso!")

Carregando os dados temporais...
Realizando Split Cronológico por TAG...


Splitting: 100%|██████████| 32/32 [00:00<00:00, 503.52it/s]


Aplicando StandardScaler (Fit no Treino, Transform no Teste)...

Gerando tensores de Treino...


Gerando janelas (Treino): 100%|██████████| 32/32 [00:02<00:00, 11.88it/s] 



Gerando tensores de Teste...


Gerando janelas (Teste): 100%|██████████| 32/32 [00:00<00:00, 90.77it/s] 



--- RESUMO DOS TENSORES ---
Treino X: (1391042, 30, 14) | Y: (1391042,) | % Classe 1: 9.65%
Teste  X: (1079059, 30, 14)  | Y: (1079059,)  | % Classe 1: 5.21%
Tensores isolados e salvos com sucesso!


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Lambda, Embedding, Concatenate, LSTM, Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# 1. Calculando pesos das classes para lidar com o desbalanceamento no treino
pesos = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(pesos))
print(f"Pesos das Classes: {class_weights}")

# 2. Definição da Entrada
# Formato: (Passos no Tempo = 30, Atributos = 14)
input_layer = Input(shape=(SEQ_LENGTH, 14), name="Input_Completo")

# 3. Funções de Fatiamento (Slicing) para as camadas Lambda
# O índice das colunas segue a lista: 
# 2 = Matricula_Operador_Hash | 3 = Id_Alarme
def slice_operador(x): return x[:, :, 2]
def slice_alarme(x): return x[:, :, 3]

def slice_numericos_e_restante(x):
    # Pega índices 0, 1 (Tag, Tipo) e 4 a 13 (Id_Criticidade até Dia_6)
    part_1 = x[:, :, 0:2]
    part_2 = x[:, :, 4:14]
    return tf.concat([part_1, part_2], axis=-1)

# Aplicando as fatias
operador_input = Lambda(slice_operador, name="Slice_Operador")(input_layer)
alarme_input = Lambda(slice_alarme, name="Slice_Alarme")(input_layer)
numericos_input = Lambda(slice_numericos_e_restante, name="Slice_Numericos")(input_layer)

# 4. Camadas de Embedding (Ajuste o output_dim conforme sua necessidade)
EMB_DIM = 8 
emb_operador = Embedding(input_dim=VOCAB_OPERADOR, output_dim=EMB_DIM, name="Emb_Operador")(operador_input)
emb_alarme = Embedding(input_dim=VOCAB_ALARME, output_dim=EMB_DIM, name="Emb_Alarme")(alarme_input)

# 5. Concatenação de todas as features em cada passo de tempo
# As saídas do embedding terão shape (Batch, 30, 8), e a parte numérica (Batch, 30, 12)
concat_layer = Concatenate(axis=-1, name="Concat_Features")([numericos_input, emb_operador, emb_alarme])

# 6. Arquitetura LSTM Profunda
lstm_1 = LSTM(64, return_sequences=True, name="LSTM_1")(concat_layer)
bn_1 = BatchNormalization(name="BatchNorm_1")(lstm_1)
drop_1 = Dropout(0.3, name="Drop_1")(bn_1)

lstm_2 = LSTM(32, return_sequences=False, name="LSTM_2")(drop_1)
bn_2 = BatchNormalization(name="BatchNorm_2")(lstm_2)
drop_2 = Dropout(0.3, name="Drop_2")(bn_2)

# 7. Camada de Saída (Classificação Binária)
output_layer = Dense(1, activation='sigmoid', name="Saida_Classificacao")(drop_2)

# Construindo o Modelo
model = Model(inputs=input_layer, outputs=output_layer)

# 8. Definição de Métricas e Compilação
metricas = [
    tf.keras.metrics.AUC(name='auc_roc'),
    tf.keras.metrics.AUC(curve='PR', name='auc_pr'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.Precision(name='precision')
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=metricas
)

model.summary()

# 9. Configurando Early Stopping para evitar Overfitting
early_stopping = EarlyStopping(
    monitor='val_auc_pr', # Focando no PR-AUC devido ao desbalanceamento
    mode='max',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# 10. Treinamento
print("\nIniciando o treinamento da rede LSTM...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=256,
    class_weight=class_weights, # Lida com o desbalanceamento
    callbacks=[early_stopping],
    verbose=1
)

5400002